### Installiamo `gurobipy`, documentazione @ https://www.gurobi.com/documentation/9.5/refman/py_python_api_details.html#sec:Python-details

In [3]:
!pip3 install gurobipy
!pip3 install numpy
!pip3 install scipy


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


# Importiamo i pacchetti necessari

In [4]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Volgiamo definire e risolvere il seguente problema

\begin{align}
  \text{maximize}\ &250x_1 + 230x_2 + 110x_3 + 350x_4 + 110x_5\\
                  &s.t.\\
                  &0\le\ x_i \lt \infty\\
                  &2x_1 + 1.5x_2 + 0.5x_3 + 2.5x_4 + 0.7x_5\leq 100.\\
                  &0.5x_1 + 0.25x_2 + 0.25x_3 + x_4 + 0.3x_5\leq 50\\
                  &x_1 + x_2 \geq 10\\
                  &x_3 = x_5\\
\end{align}

Creiamo il modello

In [5]:
model = gp.Model()

Restricted license - for non-production use only - expires 2027-11-29


Aggiungiamo le variabili con i rispettivi limiti

In [6]:
n_vars = 5
x = model.addMVar(n_vars, lb=[0] * n_vars, ub=GRB.INFINITY)
model.update()

In [7]:
print(f"Model vars:{model.getVars()}\nNum vars: {len(model.getVars())}\nVar C1: {model.getVarByName('C0')}")

Model vars:[<gurobi.Var C0>, <gurobi.Var C1>, <gurobi.Var C2>, <gurobi.Var C3>, <gurobi.Var C4>]
Num vars: 5
Var C1: <gurobi.Var C0>


Aggiungiamo i vincoli del problema

In [8]:
A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],

               [0.5, 0.25, 0.25, 1, 0.3],
               [1, 1, 0, 0, 0],
               [0, 0, 1, 0, -1]])

b = np.array([100, 50, 10, 0])
ct = model.addConstr(A[:2]@x <= b[:2])
ct2 = model.addConstr(A[-2]@x >= b[-2])
ct3 = model.addConstr(A[-1]@x == b[-1])

model.update()


Definiamo la funzione obiettivo

In [9]:
obj_coefs = np.array([250, 230, 110, 350, 110])
model.setObjective(obj_coefs @ x, GRB.MAXIMIZE)

Risolviamo il problema

In [10]:
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 5 columns and 14 nonzeros (Max)
Model fingerprint: 0x98c1760a
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]

Presolve removed 1 rows and 1 columns
Presolve time: 0.00s
Presolved: 3 rows, 4 columns, 10 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8333333e+04   2.500000e+00   0.000000e+00      0s
       1    1.7883333e+04   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.788333333e+04


Stampiamo per ogni variabile il valore nella soluzione ottima



In [11]:
for v in model.getVars():
  print('%s %g' % (v.VarName, v.X))

print('Obj: %g' % model.ObjVal)

C0 0
C1 10
C2 70.8333
C3 0
C4 70.8333
Obj: 17883.3


# Proviamo a risolvere il problema duale

In [12]:
#TODO
dual_model = gp.Model()
n_vars = 4
x = dual_model.addMVar(shape=n_vars, lb=0.0, ub=GRB.INFINITY, name="y")
dual_model.update()

c = np.array([100, 50, 10, 0], dtype=float)
A = np.array([
    [2.0,   0.5,  1.0,  0.0],   # 2y1 + 0.5y2 + 1y3       >= 250
    [1.5,   0.25, 1.0,  0.0],   # 1.5y1 + 0.25y2 + 1y3    >= 230
    [0.5,   0.25, 0.0,  1.0],   # 0.5y1 + 0.25y2 + 1y4    >= 110
    [2.5,   1.0,  0.0,  0.0],   # 2.5y1 + 1y2             >= 350
    [0.7,   0.3,  0.0, -1.0]    # 0.7y1 + 0.3y2 - 1y4     >= 110
], dtype=float)

b = np.array([250, 230, 110, 350, 110], dtype=float)




# E' possibile accedere ai vincoli ed alla funzione obiettivo

In [34]:
for c in dual_model.getConstrs():
  print(dual_model.getRow(c))

In [14]:
dual_model.getObjective()

<gurobi.LinExpr: 0.0>

# Rendiamo il problema primale privo di soluzioni ammissibili

# Rendiamo il problema primale illimitato

In [15]:
#TODO

# Problema del cammino minimo
Dato un grafo $\mathcal{G}$ e due nodi $s,\ t$ vogliamo trovare il cammino che minimiza il costo di andare da $s$ a $t$.

Creiamo un grafo costituito da $200$ nodi

In [16]:
n_nodes = 200
adj_mat = np.random.rand(n_nodes, n_nodes)
adj_mat[adj_mat > 0.02] = 0
adj_mat *= 1000
adj_mat = np.round(adj_mat)
np.fill_diagonal(adj_mat, 0)

Definiamo il nodo di partenza ed il nodo di arrivo

In [17]:
s = 0
t = 197

## Risolviamo il problema con Gurobi

Se il costo di ogni arco e' positivo, il problema si puo' formulare nel seguente modo:
\begin{align}
  \text{minimize} &\sum_{ij} c_{ij}x_{ij}\\
                  & s.t.\\
                  0 \leq\ &x \leq 1\\
                  \sum_i x_{iv} + b_v &= \sum_j x_{vj},\ \forall v \in \text{Nodes}
\end{align}
dove $b_v$ il flusso in ingresso su ogni nodo ed $x_{ij}$ la quantita' di flusso nell'arco $(i, j)$.

In [18]:
import gurobipy as gp
from gurobipy import GRB

Creiamo il modello

In [19]:
m = gp.Model('shortestPath')

Aggiungiamo le variabili e definiamo la funzione obiettivo

In [20]:
var_idxs = np.nonzero(adj_mat)
var_idxs = [(i, j) for i, j in zip(var_idxs[0], var_idxs[1])]
costs = {idx: adj_mat[idx] for idx in var_idxs}

arcs = m.addVars(var_idxs, lb = 0, ub = 1, obj=costs, name="arcs")
m.update()

In [21]:
m.getObjective()

<gurobi.LinExpr: 5.0 arcs[0,126] + 11.0 arcs[0,139] + 19.0 arcs[0,179] + 16.0 arcs[0,186] + 12.0 arcs[1,43] + 14.0 arcs[1,97] + 7.0 arcs[1,129] + 7.0 arcs[1,142] + 2.0 arcs[1,143] + arcs[1,166] + arcs[2,129] + 16.0 arcs[2,163] + 16.0 arcs[2,176] + 19.0 arcs[2,183] + 5.0 arcs[3,30] + 3.0 arcs[3,39] + 12.0 arcs[3,57] + 3.0 arcs[3,69] + 10.0 arcs[3,133] + 14.0 arcs[3,157] + 13.0 arcs[3,177] + 7.0 arcs[3,193] + 13.0 arcs[4,55] + 19.0 arcs[4,96] + 9.0 arcs[4,100] + 17.0 arcs[4,145] + 10.0 arcs[4,161] + 15.0 arcs[5,33] + 19.0 arcs[5,120] + 14.0 arcs[5,133] + 14.0 arcs[5,155] + arcs[6,16] + 3.0 arcs[6,17] + 7.0 arcs[6,77] + 16.0 arcs[6,141] + 8.0 arcs[6,175] + 20.0 arcs[7,106] + 17.0 arcs[7,154] + 8.0 arcs[7,162] + 4.0 arcs[8,7] + 6.0 arcs[8,65] + 4.0 arcs[8,81] + 10.0 arcs[8,94] + 18.0 arcs[8,115] + 11.0 arcs[8,130] + 10.0 arcs[9,74] + 5.0 arcs[9,121] + 8.0 arcs[9,156] + 5.0 arcs[9,168] + 4.0 arcs[9,169] + 6.0 arcs[10,45] + 10.0 arcs[10,103] + 8.0 arcs[10,143] + 20.0 arcs[11,66] + 9.0 arcs[1

Aggiungiamo i vincoli

In [22]:
b = np.zeros(n_nodes)
b[s] = 1
b[t] = -1

ct = m.addConstrs((arcs.sum('*', v) + b[v] == arcs.sum(v, '*') for v in range(n_nodes)))

Risolviamo il problema e stampiamo la soluzione trovata

In [23]:
m.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 200 rows, 796 columns and 1592 nonzeros (Min)
Model fingerprint: 0xe660553c
Model has 796 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 4 rows and 7 columns
Presolve time: 0.00s
Presolved: 196 rows, 789 columns, 1578 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   2.000000e+00   0.000000e+00      0s
      34    4.7000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 34 iterations and 0.00 seconds (0.00 work units)
Optimal objective  4.700000000e+01


In [24]:
print(f"Shortest path cost: {m.objVal:.4f}")
for v in var_idxs:
  if arcs[v].getAttr('X') == 1:
    print(f"{v[0]:3d} -> {v[1]:3d}: {arcs[v].getAttr('Obj'):.4f}")

Shortest path cost: 47.0000
  0 -> 139: 11.0000
  2 -> 129: 1.0000
  3 -> 157: 14.0000
 24 ->   3: 9.0000
129 -> 150: 3.0000
139 ->   2: 2.0000
150 ->  24: 3.0000
157 -> 197: 4.0000


# Misc Utili

In [25]:
a = [1,2,3,4]
b = ["a", "b", "c", "d"]

## Itertools
Implementa molte funzioni utili per iterare su collezioni di elementi

In [26]:
import itertools

In [27]:
print(list(itertools.chain(a, b)))

[1, 2, 3, 4, 'a', 'b', 'c', 'd']


In [28]:
print(list(itertools.product(a,b)))

[(1, 'a'), (1, 'b'), (1, 'c'), (1, 'd'), (2, 'a'), (2, 'b'), (2, 'c'), (2, 'd'), (3, 'a'), (3, 'b'), (3, 'c'), (3, 'd'), (4, 'a'), (4, 'b'), (4, 'c'), (4, 'd')]


## zip, enumerate

In [29]:
for a_i, b_i in zip(a, b):
  print(a_i, b_i)

1 a
2 b
3 c
4 d


In [30]:
for x in zip(a, b):
  print(x)

(1, 'a')
(2, 'b')
(3, 'c')
(4, 'd')


In [31]:
for i, a_i in enumerate(a):
  print(i, a_i)

0 1
1 2
2 3
3 4


In [32]:
for x in enumerate(a):
  print(x)

(0, 1)
(1, 2)
(2, 3)
(3, 4)


# Esercizio

Un polo industriale produce televisori utilizzando tre linee di montaggio (A, B e C) per televisori piatti, curvi e monitor ad alte prestazioni. Per stabilire la produzione per il prossimo semestre e’ necessario considerare I seguenti aspetti:

  * Dalla vendita dei tre prodotti l’azienda ricava un profitto pari a 200, 400 e 500 euro rispettivamente.

  * Nel corso dei sei mesi, la linea A potra’ essere impengata per un totale di 150 ore al mese, la linea B per 130 al mese e la linea C per 170 ore al mese. Tuttavia, esclusivamente per il quarto mese di produzione sarà possibile usufruire di ulteriori 20 ore di lavoro sulla linea A e di 10 ore in meno sulla linea C.

  * Produrre un televisore a schermo piatto richiede 18 ore sulla linea A e 7 sulla B, un televisore curvo invece ne richiede 12 sulla linea B e 25 sulla C, mentre un monitor ad alte prestazioni richiede 10 ore di lavorazione sulla linea A, 9 sulla linea B e 14 sulla linea C.

  * In un impianto separato l’azienda esegue verifiche sui prodotti ed ha una capacita’ lavorativa mensile pari a 120 ore. Per eseguire tutte le verifiche necessarie sono richieste 5 ore per un televisore a schermo piatto, 8 per uno curvo e 9 per un monitor.

  * A seguito di analisi di mercato vi sono I seguenti vincoli sulla produzione: nel secondo mese la produzione di monitor deve supereare le 10 unita’, nel terzo e quinto mese la produzione di televisori curvi deve essere almeno pari al 30% di quella di televisori piatti ed infine nel sesto mese di produzione non e’ prevista richiesta di monitor ad alte prestazioni.

  * Temendo uno sciopero dei lavoratori, la direzione considera che nel primo mese di produzione la capacita’ lavorativa dell’impianto di verifica dei prodotti sara’ ridotta al 75%.
  * la produzione complessiva sui 6 mesi di monitor ad altre prestazioni non deve superare le 40 unità

Sulla base di tali considerazioni, il problema e’ pianificiare la produzione del prossimo semestre con l’obiettivo di massimizzare il profitto dell’azienda.

In [38]:
#TODO
import gurobipy as gp
import numpy as np
from gurobipy import GRB

model = gp.Model()
n_vars = 3
x = model.addMVar(shape=(6, 3), lb=0.0, vtype=GRB.INTEGER, name="x")
model.update()
c = np.array([200, 400, 500])
prod = x @ c
model.setObjective(prod.sum(), GRB.MAXIMIZE)

cap_A = [150, 150, 150, 170, 150, 150]
cap_B = [130, 130, 130, 130, 130, 130]
cap_C = [170, 170, 170, 160, 170, 170]
cap_V = [90, 120, 120, 120, 120, 120]

for mesi in range(6):
  model.addConstr(18*x[mesi, 0] +10*x[mesi, 2] <= cap_A[mesi])
  model.addConstr(7*x[mesi, 0] + 12*x[mesi, 1] + 9*x[mesi, 2] <= cap_B[mesi])
  model.addConstr(25*x[mesi, 1] + 14*x[mesi, 2] <= cap_C[mesi])
  model.addConstr(5*x[mesi, 0].sum() + 8*x[mesi, 1].sum() + 9*x[mesi, 2].sum() <= cap_V[mesi])

model.addConstr(x[1, 2] >= 11)
model.addConstr(x[2, 1] >= 0.3 * x[2, 0])
model.addConstr(x[4, 1] >= 0.3 * x[4, 0])
model.addConstr(x[5, 2] == 0)
model.addConstr(x[:, 2].sum() <= 40)


model.optimize()


nomi = ['Piatto', 'Curvo', 'Monitor']
for t in range(6):
    for p in range(3):
        print(f"Mese {t+1}, {nomi[p]}: {x[t,p].X:.0f}")

print(f"\nProfitto ottimo: {model.ObjVal:.0f} €")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 29 rows, 18 columns and 72 nonzeros (Max)
Model fingerprint: 0xe10881fe
Model has 18 linear objective coefficients
Variable types: 0 continuous, 18 integer (0 binary)
Coefficient statistics:
  Matrix range     [3e-01, 2e+01]
  Objective range  [2e+02, 5e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 2e+02]

Found heuristic solution: objective 23500.000000
Presolve removed 11 rows and 5 columns
Presolve time: 0.00s
Presolved: 18 rows, 13 columns, 46 nonzeros
Variable types: 0 continuous, 13 integer (0 binary)
Found heuristic solution: objective 26600.000000

Root relaxation: objective 3.210647e+04, 15 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent  